# 音声のストレス判定：手法比較実験

**目標**：SNS に投稿された動画の音声・音楽コンテンツが「聴く人にストレスを与えるか」を判定する手法を探索する。

## ストレスを与えうる音声の類型

- 怒鳴り声・怒りを含む発話
- 泣き声・悲しみを表す発話
- 激しい・攻撃的な音楽（高 BPM、歪んだ音）
- 不快な環境音（工事音・叫び声）
- 不安を煽るナレーション・トーン
- 突発的な大音量（スタートル反応）

## 比較する手法

| # | 手法 | 概要 | 対象 |
|---|------|------|------|
| 1 | 音響特徴量（librosa） | エネルギー・テンポ・音程の統計量 | 任意の音声 |
| 2 | 音声感情認識（SER） | Wav2Vec2 ベースの発話感情分類 | 発話（音声） |
| 3 | 音楽感情認識（MER） | 音楽の感情的トーンの分類 | 音楽 |
| 4 | 音声テキスト化 + テキスト判定 | Whisper で文字起こし → ノートブック01 を適用 | 発話 |

## セットアップ

In [ ]:
%pip install -q librosa soundfile
%pip install -q transformers torch
%pip install -q numpy matplotlib
# 音声ファイルの生成（テスト用）
%pip install -q scipy

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib
import soundfile as sf
from scipy.io import wavfile
from scipy import signal as scipy_signal
from transformers import pipeline

matplotlib.rcParams['font.family'] = 'Hiragino Sans'

SAMPLE_RATE = 16000
print('インポート完了')

## テスト音声の生成

実験用のサンプル音声を合成音で生成する。  
実際の研究では `research/data/audio/` に実音声ファイルを置いて使用すること。

In [ ]:
import os
os.makedirs('../data/audio', exist_ok=True)

def generate_calm(duration=3.0, sr=SAMPLE_RATE) -> np.ndarray:
    """穏やかな正弦波トーン（低周波・低エネルギー）"""
    t = np.linspace(0, duration, int(sr * duration))
    wave = 0.2 * np.sin(2 * np.pi * 220 * t)  # A3（220Hz）
    wave += 0.1 * np.sin(2 * np.pi * 330 * t)  # E4（330Hz）
    # ゆっくりしたフェードイン
    fade = np.linspace(0, 1, int(sr * 0.5))
    wave[:len(fade)] *= fade
    return wave.astype(np.float32)

def generate_stressful(duration=3.0, sr=SAMPLE_RATE) -> np.ndarray:
    """ストレス性の高い音（高周波・高エネルギー・不協和音・突発音）"""
    t = np.linspace(0, duration, int(sr * duration))
    # 高周波の不協和音
    wave = 0.4 * np.sin(2 * np.pi * 880 * t)
    wave += 0.4 * np.sin(2 * np.pi * 932 * t)  # わずかにずれた音（ビート）
    # ホワイトノイズを混ぜる
    noise = 0.15 * np.random.randn(len(t))
    wave += noise
    # 突発的なスパイク（スタートル）
    spike_pos = int(sr * 1.5)
    wave[spike_pos:spike_pos+500] += 0.8 * np.random.randn(500)
    return np.clip(wave, -1, 1).astype(np.float32)

def generate_sad_tone(duration=3.0, sr=SAMPLE_RATE) -> np.ndarray:
    """悲しいトーン（低音・スロー・マイナー調）"""
    t = np.linspace(0, duration, int(sr * duration))
    # Am コード近似
    wave = 0.3 * np.sin(2 * np.pi * 110 * t)   # A2
    wave += 0.2 * np.sin(2 * np.pi * 130.8 * t) # C3
    wave += 0.15 * np.sin(2 * np.pi * 164.8 * t) # E3
    # ゆっくりしたトレモロ
    tremolo = 1 + 0.3 * np.sin(2 * np.pi * 2 * t)
    return (wave * tremolo).astype(np.float32)

audio_samples = {
    'calm':       {'audio': generate_calm(),       'category': 'positive',   'desc': '穏やかなトーン'},
    'stressful':  {'audio': generate_stressful(),  'category': 'stressful',  'desc': '不協和音・突発音'},
    'sad_tone':   {'audio': generate_sad_tone(),   'category': 'negative',   'desc': '悲しいトーン'},
}

# ファイルに保存
for name, data in audio_samples.items():
    path = f'../data/audio/{name}.wav'
    sf.write(path, data['audio'], SAMPLE_RATE)
    print(f'✓ 保存: {path}')

# 波形の可視化
fig, axes = plt.subplots(len(audio_samples), 1, figsize=(14, 7))
for ax, (name, data) in zip(axes, audio_samples.items()):
    librosa.display.waveshow(data['audio'], sr=SAMPLE_RATE, ax=ax)
    ax.set_title(f"{name} ({data['desc']})")
plt.suptitle('テスト音声波形', fontsize=12)
plt.tight_layout()
plt.show()

---
## 手法 1：音響特徴量（librosa）

音声信号から統計的特徴量を抽出し、ストレス指標を計算する。  

**使用する特徴量**

| 特徴量 | ストレスとの関連 |
|--------|----------------|
| RMS エネルギー | 高い → 音が大きい・激しい |
| スペクトル重心 | 高い → 高周波成分が多い（耳障り） |
| スペクトルフラットネス | 高い → ノイズ的（不快） |
| テンポ（BPM） | 高い → 急かされる感覚 |
| ゼロ交差率 | 高い → 音の荒さ・ノイズ |
| MFCC の分散 | 高い → 音の変動が激しい |

In [ ]:
def extract_acoustic_features(audio: np.ndarray, sr: int = SAMPLE_RATE) -> dict:
    """librosa を使って音響特徴量を抽出し、ストレススコアを計算する"""

    # --- 基本特徴量 ---
    rms = librosa.feature.rms(y=audio)[0]
    rms_mean = float(rms.mean())
    rms_max  = float(rms.max())

    spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]
    centroid_mean = float(spectral_centroid.mean()) / (sr / 2)  # 正規化

    spectral_flatness = librosa.feature.spectral_flatness(y=audio)[0]
    flatness_mean = float(spectral_flatness.mean())

    zcr = librosa.feature.zero_crossing_rate(audio)[0]
    zcr_mean = float(zcr.mean())

    # テンポ推定（高 BPM = 急かされる感覚）
    try:
        tempo, _ = librosa.beat.beat_track(y=audio, sr=sr)
        tempo_norm = min(1.0, float(tempo) / 200.0)  # 200 BPM を最大とみなす
    except Exception:
        tempo_norm = 0.0

    # MFCC の変動（感情の起伏）
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    mfcc_var = float(np.mean(np.var(mfcc, axis=1)))
    mfcc_var_norm = min(1.0, mfcc_var / 500.0)

    # --- ストレススコアの計算 ---
    # 突発的な大音量（スタートル）= 最大エネルギーと平均の比
    startle_score = min(1.0, (rms_max / (rms_mean + 1e-6) - 1.0) / 5.0)

    stress_score = (
        min(1.0, rms_mean * 5.0)    * 0.20 +  # エネルギー（音量）
        centroid_mean               * 0.20 +  # 高周波成分
        min(1.0, flatness_mean * 20)* 0.15 +  # ノイズ性
        tempo_norm                  * 0.15 +  # テンポ
        mfcc_var_norm               * 0.15 +  # 音の変動
        startle_score               * 0.15    # 突発音
    )

    return {
        'rms_mean':       round(rms_mean, 4),
        'centroid_norm':  round(centroid_mean, 3),
        'flatness':       round(flatness_mean, 4),
        'tempo_norm':     round(tempo_norm, 3),
        'mfcc_var_norm':  round(mfcc_var_norm, 3),
        'startle_score':  round(startle_score, 3),
        'stress_score':   round(stress_score, 3),
        'flagged':        stress_score > 0.4
    }

print('=== 音響特徴量 ストレス判定 ===')
acoustic_results = {}
for name, data in audio_samples.items():
    feat = extract_acoustic_features(data['audio'])
    acoustic_results[name] = feat
    flag = '🔴' if feat['flagged'] else '⚪'
    print(f"{flag} [{data['category']:12s}] score={feat['stress_score']:.3f} "
          f"rms={feat['rms_mean']:.4f} centroid={feat['centroid_norm']:.3f} "
          f"startle={feat['startle_score']:.3f}  {data['desc']}")

In [ ]:
# スペクトログラムの可視化
fig, axes = plt.subplots(len(audio_samples), 1, figsize=(14, 8))

for ax, (name, data) in zip(axes, audio_samples.items()):
    D = librosa.amplitude_to_db(np.abs(librosa.stft(data['audio'])), ref=np.max)
    img = librosa.display.specshow(D, y_axis='log', x_axis='time', sr=SAMPLE_RATE, ax=ax)
    score = acoustic_results[name]['stress_score']
    ax.set_title(f"{name} ({data['desc']}) — ストレススコア: {score:.3f}")
    plt.colorbar(img, ax=ax, format='%+2.0f dB')

plt.suptitle('手法1: スペクトログラム比較', fontsize=12)
plt.tight_layout()
plt.show()

---
## 手法 2：音声感情認識（Speech Emotion Recognition）

**モデル**：`ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition`  
（または `superb/wav2vec2-base-superb-er`）  
**感情ラベル**：`angry / calm / disgust / fearful / happy / neutral / sad / surprised`  
**特徴**：Wav2Vec2 の音声表現を使い、発話そのものから感情を推定する。  
**限界**：現時点では英語発話で最も精度が高い。日本語は研究途上。

In [ ]:
print('音声感情認識モデルを読み込み中...')
# 注: このモデルは発話（人の声）向け。合成音での精度は低い
ser_pipeline = pipeline(
    'audio-classification',
    model='superb/wav2vec2-base-superb-er',
    sampling_rate=SAMPLE_RATE
)
print('読み込み完了')

# ストレス関連感情の重みづけ（SER モデルのラベルに合わせる）
SER_STRESS_WEIGHT = {
    'ang': 1.0,    # angry
    'sad': 0.8,    # sad
    'fea': 0.9,    # fearful
    'dis': 0.7,    # disgust
    'sur': 0.2,    # surprised
    'neu': 0.0,    # neutral
    'hap': 0.0,    # happy
    'cal': 0.0,    # calm
}

def ser_stress(audio: np.ndarray) -> dict:
    try:
        # pipeline には dict 形式で渡す
        inputs = {'raw': audio, 'sampling_rate': SAMPLE_RATE}
        results = ser_pipeline(inputs)
        score_map = {r['label'].lower(): r['score'] for r in results}
        stress_score = sum(score_map.get(e, 0) * w for e, w in SER_STRESS_WEIGHT.items())
        top_emotion = max(score_map, key=score_map.get)
        return {
            'stress_score': round(stress_score, 3),
            'top_emotion':  top_emotion,
            'scores':       {e: round(s, 3) for e, s in score_map.items()},
            'flagged':      stress_score > 0.4
        }
    except Exception as e:
        return {'stress_score': 0.0, 'top_emotion': 'error', 'flagged': False, 'error': str(e)}

print('\n=== 音声感情認識（SER）===')
print('（合成音での精度は低め。実発話で試すことを推奨）')
ser_results = {}
for name, data in audio_samples.items():
    res = ser_stress(data['audio'])
    ser_results[name] = res
    flag = '🔴' if res['flagged'] else '⚪'
    print(f"{flag} [{data['category']:12s}] score={res['stress_score']:.3f} top={res['top_emotion']:10s}  {data['desc']}")

---
## 手法 3：音声テキスト化 + テキスト判定（パイプライン連携）

**構成**：`Whisper`（OpenAI）で文字起こし → ノートブック01 のテキスト判定を適用  
**特徴**：音声の内容（話している言葉）を判定できる。日本語の発話に有効。  
**限界**：音声のトーン・感情は無視される。テキスト判定の精度に依存する。

In [ ]:
print('Whisper モデルを読み込み中...')
# openai/whisper-small が日本語対応で軽量
asr_pipeline = pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-small',
    generate_kwargs={'language': 'ja', 'task': 'transcribe'}
)
print('読み込み完了')

def transcribe(audio: np.ndarray) -> str:
    try:
        result = asr_pipeline({'raw': audio, 'sampling_rate': SAMPLE_RATE})
        return result.get('text', '').strip()
    except Exception as e:
        return f'[エラー: {e}]'

print('\n=== Whisper 文字起こし ===')
print('（合成音では意味のあるテキストが得られないため、実発話ファイルで試すこと）')
for name, data in audio_samples.items():
    text = transcribe(data['audio'])
    print(f"  {name}: {repr(text)}")

print()
print('--- 実発話ファイルで使う場合 ---')
print('''
# speech.wav を文字起こしして、テキスト判定を適用する例
audio, sr = librosa.load("../data/audio/speech.wav", sr=16000)
text = transcribe(audio)
print(f"テキスト: {text}")

# ノートブック01 の sentiment_pipeline を使ってテキスト判定
# result = sentiment_pipeline(text)
# print(result)
''')

---
## 手法 4：カスタム特徴量 — 突発音・異常音検出

スタートル反応（驚愕反射）を引き起こす突発的大音量や、不快な周波数成分を検出する。  
SNS の動画コンテンツにおける「音響的いじめ」や「jumpscares」への対応を想定する。

In [ ]:
def detect_audio_hazards(audio: np.ndarray, sr: int = SAMPLE_RATE) -> dict:
    """音響的ストレス要因を多角的に検出する"""
    results = {}

    # --- 1. 突発的大音量（スタートル）---
    rms = librosa.feature.rms(y=audio, frame_length=512, hop_length=256)[0]
    rms_mean = rms.mean()
    rms_max  = rms.max()
    # 平均の3倍以上のスパイクがあれば突発音あり
    spike_mask = rms > rms_mean * 3
    has_startle = bool(spike_mask.any())
    results['startle'] = {
        'detected': has_startle,
        'max_ratio': round(float(rms_max / (rms_mean + 1e-8)), 2)
    }

    # --- 2. 不快周波数帯（3kHz〜8kHz）の卓越 ---
    # 人間の聴覚が最も敏感で、耳障りに感じやすい帯域
    fft = np.abs(librosa.stft(audio))
    freqs = librosa.fft_frequencies(sr=sr)
    discomfort_mask = (freqs >= 3000) & (freqs <= 8000)
    total_energy = fft.sum()
    discomfort_energy = fft[discomfort_mask].sum()
    discomfort_ratio = float(discomfort_energy / (total_energy + 1e-8))
    results['discomfort_freq'] = {
        'ratio': round(discomfort_ratio, 3),
        'flagged': discomfort_ratio > 0.3
    }

    # --- 3. 不協和音の検出（音程のビート感）---
    # スペクトルフラットネスの変動が大きい = 不協和な構造
    flatness = librosa.feature.spectral_flatness(y=audio)[0]
    flatness_var = float(np.var(flatness))
    results['dissonance'] = {
        'flatness_var': round(flatness_var, 5),
        'flagged': flatness_var > 0.01
    }

    # --- 総合ストレススコア ---
    stress_score = (
        (1.0 if has_startle else 0.0)              * 0.4 +
        min(1.0, discomfort_ratio * 3)             * 0.35 +
        min(1.0, flatness_var * 100)               * 0.25
    )
    results['stress_score'] = round(stress_score, 3)
    results['flagged'] = stress_score > 0.35
    return results

print('=== 音響ハザード検出 ===')
hazard_results = {}
for name, data in audio_samples.items():
    res = detect_audio_hazards(data['audio'])
    hazard_results[name] = res
    flag = '🔴' if res['flagged'] else '⚪'
    startle = '⚡突発音' if res['startle']['detected'] else ''
    discomf = '👂不快周波数' if res['discomfort_freq']['flagged'] else ''
    disson  = '🎵不協和音' if res['dissonance']['flagged'] else ''
    detected = ' '.join(filter(None, [startle, discomf, disson])) or 'なし'
    print(f"{flag} [{data['category']:12s}] score={res['stress_score']:.3f}  検出: {detected}")
    print(f"    {data['desc']}")

---
## 比較・可視化

In [ ]:
import pandas as pd

comp_data = []
for name, data in audio_samples.items():
    comp_data.append({
        'name': name,
        'category': data['category'],
        '音響特徴量': acoustic_results.get(name, {}).get('stress_score', float('nan')),
        'SER': ser_results.get(name, {}).get('stress_score', float('nan')),
        'ハザード検出': hazard_results.get(name, {}).get('stress_score', float('nan')),
    })

df_audio = pd.DataFrame(comp_data).set_index('name')

import seaborn as sns
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(
    df_audio[['音響特徴量', 'SER', 'ハザード検出']],
    annot=True, fmt='.3f', cmap='RdYlGn_r',
    vmin=0, vmax=1, ax=ax, linewidths=0.5
)
ax.set_title('音声ストレス判定：手法比較', fontsize=12)
plt.tight_layout()
plt.show()

print(df_audio)

---
## 実音声ファイルで実験する場合

In [ ]:
# research/data/audio/ に音声ファイルを置いて実行

def analyze_audio_file(path: str, label: str = '') -> None:
    """任意の音声ファイルを全手法で分析する"""
    audio, sr = librosa.load(path, sr=SAMPLE_RATE)
    duration = len(audio) / sr
    print(f'\n📁 {path} ({label}) — {duration:.1f}秒')

    # 手法1: 音響特徴量
    feat = extract_acoustic_features(audio)
    print(f'  [音響特徴量] score={feat["stress_score"]:.3f} {"🔴" if feat["flagged"] else "⚪"}')

    # 手法2: SER
    ser = ser_stress(audio)
    print(f'  [SER]       score={ser["stress_score"]:.3f} top={ser["top_emotion"]} {"🔴" if ser["flagged"] else "⚪"}')

    # 手法3: テキスト化
    text = transcribe(audio)
    print(f'  [Whisper]   "{text[:60]}"')

    # 手法4: ハザード検出
    hazard = detect_audio_hazards(audio)
    print(f'  [ハザード]   score={hazard["stress_score"]:.3f} {"🔴" if hazard["flagged"] else "⚪"}')

# 使用例
# analyze_audio_file('../data/audio/your_file.wav', label='怒りの発話サンプル')
print('analyze_audio_file() を使って任意の音声ファイルを分析できます')
print('例: analyze_audio_file("../data/audio/sample.wav", label="テストサンプル")')

---
## 考察・まとめ

### 各手法の評価

| 手法 | 得意 | 限界 |
|------|------|------|
| 音響特徴量 | 音量・音程・ノイズ的特性。音楽・環境音にも適用可 | 発話の内容・感情は取れない |
| SER | 発話のトーン・感情。怒り・悲しみの声を検出 | 英語最適化。音楽・環境音には適用不可 |
| Whisper + テキスト判定 | 発話の内容を評価できる。日本語対応 | 音のトーンは無視。文字起こしコストあり |
| ハザード検出 | 突発音・不快周波数・不協和音を直接検出 | 音楽の感情的トーンは検出できない |

### SNS 動画コンテンツへの推奨アーキテクチャ

```
動画から音声を抽出
  ↓
音響特徴量分析（常時オン・軽量）
  ↓
人声の検出（VAD: Voice Activity Detection）
  ├─ 人声あり → SER + Whisper 文字起こし + テキスト判定
  └─ 人声なし（音楽・環境音）→ 音楽感情認識 + ハザード検出
```

### 今後の実験課題

1. **日本語 SER モデル**：日本語発話の感情認識モデルの評価（`rinna/japanese-wav2vec2-base` 等）
2. **音楽ジャンル分類**：ハードコア・ノイズ音楽の自動検出
3. **マルチモーダル統合**：テキスト01 + 画像02 + 音声03 の組み合わせスコア
4. **実データ評価**：Bluesky の動画付き投稿で誤検知率を測定
5. **生理指標との相関**：心拍数・皮膚電気反応との相関でモデルを評価